# Load User Metadata

In [1]:
import pandas as pd
import numpy as np

df_users_final = pd.read_parquet("./datasets/final_outputs/df_users_final.parquet")

df_users = df_users_final.copy()
print(df_users.shape)
print(df_users.columns.tolist())
df_users_final.head()

(40000, 28)
['id', 'label', 'split', 'participation_degree', 'created_at', 'description', 'entities', 'location', 'name', 'pinned_tweet_id', 'profile_image_url', 'protected', 'url', 'username', 'verified', 'withheld', 'public_metrics.followers_count', 'public_metrics.following_count', 'public_metrics.tweet_count', 'public_metrics.listed_count', 'entities.url.urls', 'entities.description', 'entities.url', 'entities.description.urls', 'entities.description.mentions', 'entities.description.hashtags', 'entities.description.cashtags', 'withheld.country_codes']


,id,label,split,participation_degree,created_at,description,entities,location,name,pinned_tweet_id,...,public_metrics.tweet_count,public_metrics.listed_count,entities.url.urls,entities.description,entities.url,entities.description.urls,entities.description.mentions,entities.description.hashtags,entities.description.cashtags,withheld.country_codes
0,u2664730894,human,train,1726,2014-07-02 17:56:46+00:00,creative _,NaN,🎈,olawale 💨,NaN,...,1823,0,None,NaN,NaN,None,None,None,None,None
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,,NaN,🇬🇧,Grian,1.143808e+18,...,1400,448,"[{'display_url': 'youtube.com/c/grian', 'end':...",NaN,NaN,None,None,None,None,None
2,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",NaN,None,AK,NaN,...,9194,605,None,NaN,NaN,None,None,None,None,None
3,u1467973039883182090,human,train,1207,2021-12-06 21:44:04+00:00,https://t.co/Hmg5gBvd9A,NaN,None,صارا,NaN,...,146,2,None,NaN,NaN,"[{'display_url': 't.me/BiChatBot?star…', 'end'...",None,None,None,None
4,u234059290,human,train,2267,2011-01-04 19:11:39+00:00,Come for the science (genetics & cell biology)...,NaN,"Salt Lake City,UT, USA",Professor Booty PhD,1.470252e+18,...,91381,116,"[{'display_url': 'profbootyphd.wordpress.com',...",NaN,NaN,None,None,"[{'end': 129, 'start': 112, 'tag': 'BlackLives...",None,None


# Drop duplicates 

In [ ]:
df_users["id"] = df_users["id"].astype(str).str.strip()

df_users = df_users.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(df_users.shape)
print(df_users["id"].nunique())

(40000, 28)
40000


# Check for Sparsity and drop columns with over 90% missing data

In [ ]:
sparsity_df = pd.DataFrame({
    "column": df_users.columns,
    "dtype": [str(df_users[c].dtype) for c in df_users.columns],
    "missing_count": [df_users[c].isna().sum() for c in df_users.columns],
    "missing_ratio": [df_users[c].isna().mean() for c in df_users.columns],
})

sparsity_df["zero_count"] = [
    (df_users[c] == 0).sum() if pd.api.types.is_numeric_dtype(df_users[c]) else None
    for c in df_users.columns
]

sparsity_df["zero_ratio"] = [
    (df_users[c] == 0).mean() if pd.api.types.is_numeric_dtype(df_users[c]) else None
    for c in df_users.columns
]

sparsity_df["empty_string_count"] = [
    (df_users[c].astype(str).str.strip() == "").sum() if df_users[c].dtype == "object" else None
    for c in df_users.columns
]

sparsity_df["empty_string_ratio"] = [
    (df_users[c].astype(str).str.strip() == "").mean() if df_users[c].dtype == "object" else None
    for c in df_users.columns
]

sparsity_df = sparsity_df.sort_values(
    by=["missing_ratio", "zero_ratio"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

display(sparsity_df)

cols_to_drop = sparsity_df.loc[sparsity_df["missing_ratio"] > 0.80, "column"].tolist()

print("Columns to drop (>80% missing):")
print(cols_to_drop)
print("Number of columns to drop:", len(cols_to_drop))

df_users = df_users.drop(columns=cols_to_drop)

print(df_users.shape)

,column,dtype,missing_count,missing_ratio,zero_count,zero_ratio,empty_string_count,empty_string_ratio
0,entities,float64,40000,1.000000,0.0,0.000000,NaN,NaN
1,withheld,float64,40000,1.000000,0.0,0.000000,NaN,NaN
2,entities.description,float64,40000,1.000000,0.0,0.000000,NaN,NaN
3,entities.url,float64,40000,1.000000,0.0,0.000000,NaN,NaN
4,withheld.country_codes,object,39997,0.999925,NaN,NaN,0.0,0.000000
5,entities.description.cashtags,object,39807,0.995175,NaN,NaN,0.0,0.000000
6,entities.description.urls,object,35593,0.889825,NaN,NaN,0.0,0.000000
7,entities.description.hashtags,object,32302,0.807550,NaN,NaN,0.0,0.000000
8,entities.description.mentions,object,29467,0.736675,NaN,NaN,0.0,0.000000
9,pinned_tweet_id,float64,21747,0.543675,0.0,0.000000,NaN,NaN


Columns to drop (>80% missing):
['entities', 'withheld', 'entities.description', 'entities.url', 'withheld.country_codes', 'entities.description.cashtags', 'entities.description.urls', 'entities.description.hashtags']
Number of columns to drop: 8
(40000, 20)


# Inspect other relevant columns with high missing content ratio to see if it can be dropped

- mentions covers users with accounts mentioned in their description, however due to the overall sparsity of the column it should still be dropped
- urls column does not provide much meaningful data and can be dropped
- pinned_tweet_id can be more useful as a binary feature

In [ ]:
cols_to_check = [
    "entities.description.mentions",
    "entities.url.urls",
]

for col in cols_to_check:
    if col in df_users.columns:
        print(f"\n=== {col} ===")
        print("dtype:", df_users[col].dtype)
        print("missing ratio:", df_users[col].isna().mean())
        print("non-missing count:", df_users[col].notna().sum())

        display(df_users.loc[df_users[col].notna(), [col]].head(10))
    else:
        print(f"\n{col} not found in df_users.columns")

df_users["has_pinned_tweet"] = df_users["pinned_tweet_id"].notna().astype(int)

df_users = df_users.drop(
    columns=[
        c for c in [
            "entities.description.mentions",
            "entities.url.urls",
        ] if c in df_users.columns
    ]
)



=== entities.description.mentions ===
dtype: object
missing ratio: 0.736675
non-missing count: 10533


,entities.description.mentions
5,"[{'end': 51, 'start': 35, 'username': 'parccie..."
14,"[{'end': 21, 'start': 13, 'username': 'Walmart'}]"
19,"[{'end': 26, 'start': 13, 'username': 'thought..."
31,"[{'end': 34, 'start': 26, 'username': 'Orioles'}]"
32,"[{'end': 49, 'start': 38, 'username': 'UCSanDi..."
33,"[{'end': 18, 'start': 7, 'username': 'cornellc..."
35,"[{'end': 20, 'start': 4, 'username': 'ArtsBusi..."
36,"[{'end': 32, 'start': 23, 'username': 'hashnod..."
39,"[{'end': 102, 'start': 86, 'username': 'Unconf..."
45,"[{'end': 92, 'start': 82, 'username': 'cheaper..."



=== entities.url.urls ===
dtype: object
missing ratio: 0.409075
non-missing count: 23637


,entities.url.urls
1,"[{'display_url': 'youtube.com/c/grian', 'end':..."
4,"[{'display_url': 'profbootyphd.wordpress.com',..."
5,"[{'display_url': 'icmol.es', 'end': 23, 'expan..."
7,"[{'display_url': 'nikhilgopal.com', 'end': 23,..."
8,"[{'display_url': 'TheDailyCrossFit.com', 'end'..."
11,"[{'display_url': 'instagram.com/amnimanimals',..."
12,"[{'display_url': 'pcb.com.pk/player/fakhar-…',..."
13,[{'display_url': 'youtube.com/channel/UCATRl…'...
15,"[{'display_url': 'vintagecomputers.code.blog',..."
17,"[{'display_url': 'dombecklab.org', 'end': 23, ..."


# Fill empty text columns

In [ ]:
text_cols = [
    "description",
    "location",
    "name",
    "username",
    "url",
    "profile_image_url",
]

for col in text_cols:
    if col in df_users.columns:
        df_users[col] = df_users[col].fillna("unknown").astype(str).str.strip()
        df_users[col] = df_users[col].replace("", "unknown")

# Fill Empty Numeric columns

In [ ]:
numeric_cols = [
    "participation_degree",
    "public_metrics.followers_count",
    "public_metrics.following_count",
    "public_metrics.tweet_count",
    "public_metrics.listed_count",
]

for col in numeric_cols:
    if col in df_users.columns:
        df_users[col] = pd.to_numeric(df_users[col], errors="coerce").fillna(0)

# Convert created_at to datetime

In [7]:
df_users["created_at"] = pd.to_datetime(df_users["created_at"], errors="coerce", utc=True)

# Convert booleans to int

In [ ]:
bool_cols = ["protected", "verified"]

for col in bool_cols:
    df_users[col] = df_users[col].astype("boolean").fillna(False).astype(int)

# Account Age Features

In [9]:
REF_TIME = pd.Timestamp("2022-04-01", tz="UTC")
df_users["account_age_days"] = (REF_TIME - df_users["created_at"]).dt.days
df_users["account_age_days"] = df_users["account_age_days"].fillna(0).clip(lower=0)

# Rename metrics into easier column names

In [10]:
rename_map = {
    "public_metrics.followers_count": "followers_count",
    "public_metrics.following_count": "following_count",
    "public_metrics.tweet_count": "tweet_count",
    "public_metrics.listed_count": "listed_count",
}

existing_rename = {k: v for k, v in rename_map.items() if k in df_users.columns}
df_users = df_users.rename(columns=existing_rename)

df_users.head()

,id,label,split,participation_degree,created_at,description,location,name,pinned_tweet_id,profile_image_url,protected,url,username,verified,followers_count,following_count,tweet_count,listed_count,has_pinned_tweet,account_age_days
0,u2664730894,human,train,1726,2014-07-02 17:56:46+00:00,creative _,🎈,olawale 💨,NaN,https://pbs.twimg.com/profile_images/147837638...,0,unknown,wale_io,0,123,1090,1823,0,0,2829
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,unknown,🇬🇧,Grian,1.143808e+18,https://pbs.twimg.com/profile_images/100773461...,0,https://t.co/V3FyRYAsvK,GrianMC,0,238254,293,1400,448,1,3147
2,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",unknown,AK,NaN,https://pbs.twimg.com/profile_images/145119163...,0,unknown,ak92501,0,45541,1206,9194,605,0,2895
3,u1467973039883182090,human,train,1207,2021-12-06 21:44:04+00:00,https://t.co/Hmg5gBvd9A,unknown,صارا,NaN,https://pbs.twimg.com/profile_images/146934055...,0,unknown,_3rw_,0,1573,1688,146,2,0,115
4,u234059290,human,train,2267,2011-01-04 19:11:39+00:00,Come for the science (genetics & cell biology)...,"Salt Lake City,UT, USA",Professor Booty PhD,1.470252e+18,https://pbs.twimg.com/profile_images/551855299...,0,https://t.co/pKcvVO96Yk,ProfBootyPhD,0,4694,4739,91381,116,1,4104


# Profile completeness features

In [11]:

df_users["has_description"] = (df_users["description"].str.lower() != "unknown").astype(int)

df_users["has_location"] = (df_users["location"].str.lower() != "unknown").astype(int)

df_users["has_url"] = (df_users["url"].str.lower() != "unknown").astype(int)

df_users["has_profile_image"] = (df_users["profile_image_url"].str.lower() != "unknown").astype(int)

df_users["has_pinned_tweet"] = df_users["pinned_tweet_id"].notna().astype(int)

df_users["has_default_avatar"] = (
    df_users["profile_image_url"].str.contains("default_profile", case=False, na=False)
).astype(int)
print("default avatar rate:", df_users["has_default_avatar"].mean())

default avatar rate: 0.03205


In [12]:
profile_parts = [
    c for c in [
        "has_description",
        "has_location",
        "has_url",
        "has_profile_image",
        "has_pinned_tweet",
        "verified",
    ] if c in df_users.columns
]

if profile_parts:
    df_users["profile_completeness_score"] = df_users[profile_parts].sum(axis=1)

# Ratio Features

In [ ]:
df_users["log_followers_following_ratio"] = (
    np.log1p(df_users["followers_count"]) - np.log1p(df_users["following_count"])
)

df_users["log_following_followers_ratio"] = (
    np.log1p(df_users["following_count"]) - np.log1p(df_users["followers_count"])
)

df_users["log_tweets_followers_ratio"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["followers_count"])
)

df_users["log_tweets_following_ratio"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["following_count"])
)

df_users["log_listed_followers_ratio"] = (
    np.log1p(df_users["listed_count"]) - np.log1p(df_users["followers_count"])
)

df_users["log_normalized_followers"] = (
    np.log1p(df_users["followers_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_following"] = (
    np.log1p(df_users["following_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_tweets"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_listed"] = (
    np.log1p(df_users["listed_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["average_tweets_per_day"] = df_users["tweet_count"] / (df_users["account_age_days"] + 1)

df_users["log_tweets_per_day_relative_to_age"] = (
    np.log1p(df_users["average_tweets_per_day"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_tweet_engagement_ratio"] = (
    np.log1p(df_users["tweet_count"]) -
    np.log1p(df_users["followers_count"] + df_users["following_count"] + 1)
)

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

idx_train, idx_tmp = train_test_split(
    df_users.index, test_size=0.2,
    stratify=df_users["label"], random_state=RANDOM_STATE
)
idx_val, idx_test = train_test_split(
    idx_tmp, test_size=0.5,
    stratify=df_users.loc[idx_tmp, "label"], random_state=RANDOM_STATE
)

df_users["model_split"] = "train"
df_users.loc[idx_val, "model_split"] = "val"
df_users.loc[idx_test, "model_split"] = "test"

print(df_users.groupby("model_split")["label"].value_counts(normalize=True).unstack())

label         bot  human
model_split             
test         0.39   0.61
train        0.39   0.61
val          0.39   0.61


# Composite Indicators

In [ ]:
train_mask = df_users["model_split"] == "train"
print(f"Fitting thresholds on {train_mask.sum():,} training users")

high_activity_threshold = df_users.loc[train_mask, "average_tweets_per_day"].quantile(0.90)

df_users["high_activity_low_followers"] = (
    (df_users["average_tweets_per_day"] > high_activity_threshold) &
    (df_users["followers_count"] < 100)
).astype(int)

df_users["high_tweet_low_age"] = (
    (df_users["average_tweets_per_day"] > high_activity_threshold) &
    (df_users["account_age_days"] < 365)
).astype(int)

followers_threshold = df_users.loc[train_mask, "followers_count"].quantile(0.10)

df_users["is_low_follower_high_activity"] = (
    (df_users["followers_count"] < followers_threshold) &
    (df_users["average_tweets_per_day"] > high_activity_threshold)
).astype(int)

tweet_threshold = df_users.loc[train_mask, "tweet_count"].quantile(0.90)

df_users["new_account_high_tweets"] = (
    (df_users["account_age_days"] < 365) &
    (df_users["tweet_count"] > tweet_threshold)
).astype(int)

df_users["follows_more_than_followed"] = (
    df_users["following_count"] > df_users["followers_count"]
).astype(int)

df_users["is_new_with_verified"] = (
    (df_users["account_age_days"] < 365) & (df_users["verified"] == 1)
).astype(int)


df_users["account_age_category"] = pd.cut(
    df_users["account_age_days"],
    bins=[0, 365, 730, 1825, float("inf")],
    labels=["Very New", "New", "Established", "Old"]
).cat.codes

Fitting thresholds on 32,000 training users


# Text Based Features

In [ ]:
_desc = df_users["description"].where(df_users["has_description"] == 1, "")

df_users["description_length"] = _desc.str.len()
df_users["description_word_count"] = _desc.str.split().str.len().fillna(0).astype(int)
df_users["username_length"] = df_users["username"].str.len()

df_users["has_url_in_description"] = df_users["description"].apply(
    lambda x: int("http" in x.lower() or "www" in x.lower())
)

df_users["description_special_chars_count"] = df_users["description"].apply(
    lambda x: sum(1 for c in x if c in "!@#$%^&*") if isinstance(x, str) else 0
)

df_users["description_uppercase_ratio"] = df_users["description"].apply(
    lambda x: sum(c.isupper() for c in x) / len(x) if len(x) > 0 else 0
)

df_users["username_special_chars_ratio"] = df_users["username"].apply(
    lambda x: sum(1 for c in x if not c.isalnum()) / len(x) if len(x) > 0 else 0
)

df_users["username_digit_count"] = df_users["username"].str.count(r"\d")

df_users["username_upper_ratio"] = df_users["username"].apply(
    lambda x: sum(c.isupper() for c in x) / len(x) if len(x) > 0 else 0
)

df_users["username_underscore_count"] = df_users["username"].str.count("_")

# Other Feature columns

In [ ]:
df_users["follower_following_disparity"] = abs(
    df_users["followers_count"] - df_users["following_count"]
)

df_users["follower_following_disparity"] = abs(
    df_users["followers_count"] - df_users["following_count"]
)

spam_words = [
    'win', 'free', 'offer', 'click', 'buy', 'subscribe', 
    'act now', 'apply now', 'call now', 'don’t hesitate', 
    'for only', 'get started now', 'limited time', 'great offer', 
    'instant', 'now only', 'offer expires', 'once in a lifetime', 
    'order now', 'order today', 'special promotion', 'urgent', 
    'while supplies last', 'bonus', 'all new', 'amazing', 
    'certified', 'congratulations', 'fantastic deal', 'for free', 
    'guaranteed', 'outstanding value', 'risk free', 
    'satisfaction guaranteed', 'free!', 'free trial', 'free consultation', 
    'free gift', 'free membership', 'free offer', 'free preview', 
    'free sample', 'free quote', 'sign up free today', 'deal', 
    'giving away', 'no obligation', 'no strings attached', 'offer', 
    'prize', 'trial', 'unlimited', 'what are you waiting for?', 
    'win', 'winner', 'you’re a winner!', 'won', 'you have been selected', 
    '#1', '100% free', '100% satisfied', '50% off', 
    'one hundred percent guaranteed', 'click below', 'click here', 
    'increase sales', 'increase your sales', 'opt in', 'open', 'sale', 
    'sales', 'subscribe', 'chance', 'sample', 'satisfaction', 'solution', 
    'success', 'cards accepted', 'full refund', 'affordable', 
    'bargain', 'best price', 'cash', 'cash bonus', 'cheap', 
    'credit', 'discount', 'for just $', 'lowest price', 'save big money', 
    'why pay more?', 'buy', 'as seen on', 'buy direct', 'clearance', 
    'order', '$$$', 'marketing solutions', 'join millions', 
    'name brand', 'no questions asked', 'giving it away', 
    'best rates', 'compare', 'drastically reduced'
]

df_users["description_spam_score"] = df_users["description"].apply(
    lambda x: sum(word in x.lower() for word in spam_words) if isinstance(x, str) else 0
)

In [ ]:
df_users.head()

,id,label,split,participation_degree,created_at,description,location,name,pinned_tweet_id,profile_image_url,...,username_length,has_url_in_description,description_special_chars_count,description_uppercase_ratio,username_special_chars_ratio,username_digit_count,username_upper_ratio,username_underscore_count,follower_following_disparity,description_spam_score
0,u2664730894,human,train,1726,2014-07-02 17:56:46+00:00,creative _,🎈,olawale 💨,NaN,https://pbs.twimg.com/profile_images/147837638...,...,7,0,0,0.000000,0.142857,0,0.000000,1,967,0
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,unknown,🇬🇧,Grian,1.143808e+18,https://pbs.twimg.com/profile_images/100773461...,...,7,0,0,0.000000,0.000000,0,0.428571,0,237961,0
2,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",unknown,AK,NaN,https://pbs.twimg.com/profile_images/145119163...,...,7,0,0,0.000000,0.000000,5,0.000000,0,44335,1
3,u1467973039883182090,human,train,1207,2021-12-06 21:44:04+00:00,https://t.co/Hmg5gBvd9A,unknown,صارا,NaN,https://pbs.twimg.com/profile_images/146934055...,...,5,1,0,0.130435,0.400000,1,0.000000,2,115,0
4,u234059290,human,train,2267,2011-01-04 19:11:39+00:00,Come for the science (genetics & cell biology)...,"Salt Lake City,UT, USA",Professor Booty PhD,1.470252e+18,https://pbs.twimg.com/profile_images/551855299...,...,12,0,4,0.067114,0.000000,0,0.333333,0,45,0


# TF-IDF on Description Column

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df_users["description"] = df_users["description"].fillna("unknown").astype(str)

tfidf = TfidfVectorizer(
    max_features=300,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

train_mask = df_users["model_split"] == "train"

tfidf.fit(df_users.loc[train_mask, "description"])
description_tfidf = tfidf.transform(df_users["description"])
print(f"TF-IDF vocabulary: {len(tfidf.get_feature_names_out())} terms from {train_mask.sum():,} training descriptions")

tfidf_feature_names = [f"desc_tfidf_{f}" for f in tfidf.get_feature_names_out()]

df_description_tfidf = pd.DataFrame(
    description_tfidf.toarray(),
    columns=tfidf_feature_names,
    index=df_users.index
)

df_users = pd.concat([df_users, df_description_tfidf], axis=1)
print(df_users.shape)

TF-IDF vocabulary: 300 terms from 32,000 training descriptions
(40000, 358)


# Drop Raw Text Columns

In [20]:
cols_to_drop = [
    "created_at",
    "username",
    "name",
    "description",
    "location",
    "url",
    "profile_image_url",
    "entities.description.urls",
    "entities.description.mentions",
    "entities.description.hashtags",
    "entities.description.cashtags",
]

df_users_model = df_users.drop(columns=[c for c in cols_to_drop if c in df_users.columns])

# Merge user metadata FE with tweets FE (file from FE_tweets.ipynb)

In [ ]:
df_tweets_model = pd.read_parquet("./datasets/final_outputs/df_tweets_model.parquet")
print(df_tweets_model.shape)

df_merged = df_users_model.merge(
    df_tweets_model,
    on="id",
    how="left"
)

tweet_feature_cols = [c for c in df_tweets_model.columns if c != "id"]
df_merged[tweet_feature_cols] = df_merged[tweet_feature_cols].fillna(0)

n_no_tweets = df_merged["has_tweets"].eq(0).sum()
print(f"Users with no tweets: {n_no_tweets:,} / {len(df_merged):,}")
assert df_merged[tweet_feature_cols].isna().sum().sum() == 0

print(df_merged.shape)

df_merged.to_parquet("./datasets/final_outputs/df_user_meta_full.parquet", index=False)

print(df_merged.shape)

(38771, 40)
Users with no tweets: 1,229 / 40,000
(40000, 390)
(40000, 390)


In [22]:
display(df_merged.head())

,id,label,split,participation_degree,pinned_tweet_id,protected,verified,followers_count,following_count,tweet_count,...,avg_retweets_per_tweet,avg_replies_per_tweet,avg_quotes_per_tweet,engagement_rate,pct_allow_reply,fc_reply,num_unique_langs,pct_geo_tweets,has_reply_inconsistency,reply_inconsistency_count
0,u2664730894,human,train,1726,NaN,0,0,123,1090,1823,...,2735.176176,0.088088,0.003003,2735.506507,0.001001,88.00,16.0,0.0,1.0,5.0
1,u1679822588,bot,train,1127,1.143808e+18,0,0,238254,293,1400,...,278.911558,16.394975,0.869347,600.325628,0.000000,0.00,11.0,0.0,1.0,4.0
2,u2465283662,bot,test,3346,NaN,0,0,45541,1206,9194,...,93.491918,0.607759,1.496228,202.609914,0.001078,564.00,15.0,0.0,1.0,54.0
3,u1467973039883182090,human,train,1207,NaN,0,0,1573,1688,146,...,0.980435,0.939130,0.008696,5.273913,0.010870,86.40,4.0,0.0,1.0,89.0
4,u234059290,human,train,2267,1.470252e+18,0,0,4694,4739,91381,...,526.345060,0.334581,0.020958,529.062126,0.008982,37.25,9.0,0.0,1.0,164.0
